## Algo, Data & Trading Assignment

In [25]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [26]:
data_orig = pd.read_csv(r'C:\Users\Valeriu\Downloads\data.csv')

In [27]:
data_orig

,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7
0,1,32400.000000,99.98,99.980000,99.980000,5,5,0
1,1,32406.147188,100.00,99.980000,99.980000,7,3,0
2,1,32409.115557,100.02,100.004000,99.996000,4,6,1
3,1,32471.025717,100.04,100.021333,100.018667,6,4,1
4,1,32473.743952,100.02,100.042000,100.038000,8,2,1
...,...,...,...,...,...,...,...,...
102192,100,62630.991830,91.30,91.320000,91.320000,3,7,0
102193,100,62636.683584,91.28,91.300000,91.300000,3,7,1
102194,100,62725.025716,91.30,91.280000,91.280000,5,5,1
102195,100,62822.717694,91.28,91.300000,91.300000,4,6,0


Column 0 -> Day of the trade
\n Column 1 -> Time of the trade in seconds after midnigt
Column 2 -> Fundamental Value v(d,t)
Column 3 -> Ask
Column 4 -> Bid
Column 5 -> Number of Buys(d,t)
Column 6 -> Number of Sells(d,t)
Column 7 -> Signal

In [28]:
columns = ['Day', 'Time(h)', 'Value', 'Ask', 'Bid', '#Buys', '#Sells', 'Sig']

data = data_orig.copy()

data.columns = columns

data

,Day,Time(h),Value,Ask,Bid,#Buys,#Sells,Sig
0,1,32400.000000,99.98,99.980000,99.980000,5,5,0
1,1,32406.147188,100.00,99.980000,99.980000,7,3,0
2,1,32409.115557,100.02,100.004000,99.996000,4,6,1
3,1,32471.025717,100.04,100.021333,100.018667,6,4,1
4,1,32473.743952,100.02,100.042000,100.038000,8,2,1
...,...,...,...,...,...,...,...,...
102192,100,62630.991830,91.30,91.320000,91.320000,3,7,0
102193,100,62636.683584,91.28,91.300000,91.300000,3,7,1
102194,100,62725.025716,91.30,91.280000,91.280000,5,5,1
102195,100,62822.717694,91.28,91.300000,91.300000,4,6,0


In [29]:
print(data['Time(h)'].unique()[:10])

[32400.         32406.14718775 32409.11555723 32471.02571743
 32473.74395177 32487.49288034 32557.31753763 32595.66784609
 32613.77353912 32615.07621169]


In [30]:
# 1. Go back to your ORIGINAL raw seconds column
# Let's assume it was called 'Time(s)'
hours_float = pd.to_numeric(data['Time(h)'], errors='coerce') / 3600

# 2. Extract components
hours = np.floor(hours_float)
minutes_float = (hours_float - hours) * 60
minutes = np.floor(minutes_float)
seconds = np.round((minutes_float - minutes) * 60, 0) # Round to whole seconds

# 3. Create the string
data['Time(h)'] = (
    hours.astype(int).astype(str).str.zfill(2) + ":" + 
    minutes.astype(int).astype(str).str.zfill(2) + ":" + 
    seconds.astype(int).astype(str).str.zfill(2)
)

In [31]:
data

,Day,Time(h),Value,Ask,Bid,#Buys,#Sells,Sig
0,1,09:00:00,99.98,99.980000,99.980000,5,5,0
1,1,09:00:06,100.00,99.980000,99.980000,7,3,0
2,1,09:00:09,100.02,100.004000,99.996000,4,6,1
3,1,09:01:11,100.04,100.021333,100.018667,6,4,1
4,1,09:01:14,100.02,100.042000,100.038000,8,2,1
...,...,...,...,...,...,...,...,...
102192,100,17:23:51,91.30,91.320000,91.320000,3,7,0
102193,100,17:23:57,91.28,91.300000,91.300000,3,7,1
102194,100,17:25:25,91.30,91.280000,91.280000,5,5,1
102195,100,17:27:03,91.28,91.300000,91.300000,4,6,0


In [32]:
value_shift_1 = data['Value'].shift(-1)

value_shift_1
data['Value'] - value_shift_1

0        -0.02
1        -0.02
2        -0.02
3         0.02
4        -0.02
          ... 
102192    0.02
102193   -0.02
102194    0.02
102195   -0.02
102196     NaN
Name: Value, Length: 102197, dtype: float64

# Difference is always 0.02

In [33]:
Total_Buys = []
Total_Sells = []

for x in range(len(data['Sig'])):
    if data['Sig'].iloc[x] == 1:
        buy_pi = data['Value'].iloc[x] - data['Ask'].iloc[x]
        Total_Buys.append(buy_pi)
    else:
        sell_pi = data['Bid'].iloc[x] - data['Value'].iloc[x]
        Total_Sells.append(sell_pi)

In [34]:
Total_Buys = pd.DataFrame(Total_Buys)

Total_Sells = pd.DataFrame(Total_Sells)

print(Total_Sells.describe())
print(Total_Buys.describe())

                  0
count  51504.000000
mean       0.003282
std        0.019616
min       -0.030000
25%       -0.020367
50%        0.018871
75%        0.019554
max        0.020000
                  0
count  50693.000000
mean       0.003173
std        0.019637
min       -0.040000
25%       -0.020370
50%        0.018859
75%        0.019554
max        0.020000


In [35]:
count_buys = Total_Buys.count()
count_sells = Total_Sells.count()

pi_per_short = Total_Sells.mean()
pi_per_long = Total_Buys.mean()

hist_total_pi = (pi_per_long*count_buys + pi_per_short*count_sells) * 10000

hist_total_pi

0    3.298589e+06
dtype: float64

In [36]:
data['epsilon'] = data.groupby('Day')['Value'].diff()


In [37]:
innovations = data[data['epsilon'] != 0].copy()

up   = innovations[innovations['epsilon'] > 0]
tau_up = (up['Sig'] == 1).mean()


down = innovations[innovations['epsilon'] < 0]
tau_down = (down['Sig'] == 0).mean()

tau = (tau_up + tau_down) / 2

print(f"τ from up moves:   {tau_up:.4f}")
print(f"τ from down moves: {tau_down:.4f}")
print(f"τ average:         {(tau):.4f}")

τ from up moves:   0.5932
τ from down moves: 0.6003
τ average:         0.5968


In [38]:
sigma = 0.02

trades_per_day = 1000

net_edge_per_share = sigma * (2*tau - 1) * (9/10)  
P_max = net_edge_per_share * 10000 * trades_per_day * 100
print(f"Max Profit = {P_max}")

Max Profit = 3483376.9166987403


In [40]:
net_edge_per_share_2 = sigma * (2*tau - 1) * 8/10

P_max_2 = net_edge_per_share_2 * 10000 * trades_per_day * 100

print(f"Max Profit, case 2 = {P_max_2}")

Max Profit, case 2 = 3096335.037065547


In [42]:
net_edge_per_share_3 = sigma * (2*tau - 1) * 7/10

P_max_3 = net_edge_per_share_3 * 10000 * trades_per_day * 100

print(f"Max Profit, case 2 = {P_max_3}")

Max Profit, case 2 = 2709293.157432353


In [44]:
net_edge_per_share_4 = sigma * (2*tau - 1) * 6/10

P_max_4 = net_edge_per_share_4 * 10000 * trades_per_day * 100

print(f"Max Profit, case 2 = {P_max_4}")

Max Profit, case 2 = 2322251.27779916


There will be 3 people buying it, if 4 people buy it your expected profit is lower than 2.5mils and therefore it does not make any sense to buy it for 2.5mils